# Natural frequencies

The lowest natural frequencies and modes of a thin isotropic cylinder
supported at both ends. The example assembles the stiffness matrix of
`update_KC0` and the consistent mass matrix of `update_M`, and solves the
generalized eigenvalue problem with a verified solver. The same model is
checked by the test `tests/test_natural_frequency.py`.

**Theory.** The natural frequencies $\omega_n$ and the modes
$\boldsymbol{\phi}$ solve, see [Analyses](../analyses.rst),

$$
\left( \boldsymbol{K}_{C0} - \omega_n^2 \boldsymbol{M} \right)
\boldsymbol{\phi} = \boldsymbol{0}
$$

with $\boldsymbol{K}_{C0}$ positive definite once the rigid-body motions are
constrained and $\boldsymbol{M}$ the consistent mass matrix, which includes
the rotary inertia of the rotations of the normal, see
[Element matrices and internal forces](../matrices.rst). The frequencies
of a closed cylinder come in pairs of equal values, the modes
$\cos(n\theta)$ and $\sin(n\theta)$ of the same number $n$ of
circumferential waves, and the lowest pairs lie close to each other
[Leissa1973]. An iterative eigensolver can then miss or swap eigenvalues,
which is why the function `lowest_frequencies` of
`tests/natural_frequency_solver.py` is used: it computes the eigenvalues
with ARPACK in shift-invert mode, with the shift $-1$ below the lowest
eigenvalue and a fixed starting vector, checks the residual of each
eigenpair, and, by Sylvester's law of inertia, counts the negative pivots
of an $\boldsymbol{L}\boldsymbol{D}\boldsymbol{L}^\top$ factorization of
$\boldsymbol{K} - s \boldsymbol{M}$, with $s$ slightly above the highest
eigenvalue returned, to verify that none is missing.

In [1]:
import os
import sys
import numpy as np
from numpy import isclose, pi
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from composites import isotropic_plate

from bfsccylinder import (BFSCCylinder, update_KC0, update_M, DOF, DOUBLE, INT,
                          KC0_SPARSE_SIZE, KG_SPARSE_SIZE, M_SPARSE_SIZE)
from bfsccylinder.quadrature import get_points_weights
from bfsccylinder.utils import assign_constant_ABD

# the verified eigensolver shared with the tests
sys.path.insert(0, os.path.join('..', 'tests'))
from natural_frequency_solver import lowest_frequencies

from IPython.display import Markdown


def table(rows, fmt='{:.5g}'):
    """Markdown table of a list of dicts with the same keys"""
    keys = list(rows[0])
    cell = lambda v: fmt.format(v) if isinstance(v, float) else str(v)
    lines = ['| ' + ' | '.join(keys) + ' |', '|' + '---|'*len(keys)]
    lines += ['| ' + ' | '.join(cell(r[k]) for k in keys) + ' |' for r in rows]
    return Markdown('\n'.join(lines))

## Geometry, material and mesh

$R = 0.4$ m, $L = 0.8$ m, $h = 1$ mm, aluminium, $E = 70$ GPa,
$\nu = 0.33$ and $\rho = 2700$ kg/m$^3$; 21 nodes along the axis and 56
around the circumference.

In [2]:
L = 0.8  # m
R = 0.4  # m
b = 2*pi*R  # m
E = 70e9  # Pa
nu = 0.33
h = 0.001  # m
rho = 2.7e3  # kg/m3
prop = isotropic_plate(thickness=h, E=E, nu=nu)

nx = 21
ny = 56

# node numbers on a grid of nx by ny + 1, the last column closing the cylinder
nids = 1 + np.arange(nx*(ny+1))
nids_mesh = nids.reshape(nx, ny+1)
nids_mesh[:, -1] = nids_mesh[:, 0]
nids = np.unique(nids_mesh)
nid_pos = dict(zip(nids, np.arange(len(nids))))

# coordinates of the nodes on the developed mid-surface
xlin = np.linspace(0, L, nx)
ylin = np.linspace(0, b, ny+1)[:-1]
xmesh, ymesh = np.meshgrid(xlin, ylin, indexing='ij')
x = xmesh.flatten()
y = ymesh.flatten()

# the four nodes of each element, counterclockwise from (x_i, y_j)
n1s = nids_mesh[:-1, :-1].flatten()
n2s = nids_mesh[1:, :-1].flatten()
n3s = nids_mesh[1:, 1:].flatten()
n4s = nids_mesh[:-1, 1:].flatten()
num_elements = len(n1s)
N = DOF*nx*ny
print('nodes:', nx*ny, ' elements:', num_elements, ' degrees of freedom:', N)

nodes: 1176  elements: 1120  degrees of freedom: 11760


## Stiffness and mass matrices

The mass matrix needs the thickness `h` and the density `rho` of each
element; `update_M` writes its entries at the positions `init_k_M`.

In [3]:
nint = 4
points, weights = get_points_weights(nint=nint)

elements = []
for k, (n1, n2, n3, n4) in enumerate(zip(n1s, n2s, n3s, n4s)):
    shell = BFSCCylinder(nint)
    shell.n1, shell.n2, shell.n3, shell.n4 = n1, n2, n3, n4
    shell.c1 = DOF*nid_pos[n1]
    shell.c2 = DOF*nid_pos[n2]
    shell.c3 = DOF*nid_pos[n3]
    shell.c4 = DOF*nid_pos[n4]
    shell.R = R
    shell.lex = L/(nx - 1)
    shell.ley = b/ny
    shell.h = h
    shell.rho = rho
    assign_constant_ABD(shell, prop)
    shell.init_k_KC0 = k*KC0_SPARSE_SIZE
    shell.init_k_KG = k*KG_SPARSE_SIZE
    shell.init_k_M = k*M_SPARSE_SIZE
    elements.append(shell)

Kr = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
Kc = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
Kv = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=DOUBLE)
for shell in elements:
    update_KC0(shell, points, weights, Kr, Kc, Kv)
KC0 = coo_matrix((Kv, (Kr, Kc)), shape=(N, N)).tocsc()

Mr = np.zeros(M_SPARSE_SIZE*num_elements, dtype=INT)
Mc = np.zeros(M_SPARSE_SIZE*num_elements, dtype=INT)
Mv = np.zeros(M_SPARSE_SIZE*num_elements, dtype=DOUBLE)
for shell in elements:
    update_M(shell, Mr, Mc, Mv)
M = coo_matrix((Mv, (Mr, Mc)), shape=(N, N)).tocsc()

## Boundary conditions


Both ends are simply supported: $u$, $v$ and $w$ are constrained at the
nodes of $x = 0$ and $x = L$. Between two nodes of an end, $w$ is the cubic
of the nodal $w$ and $w_{,y}$, so with $w_{,y}$ free $w = 0$ holds exactly
only at the nodes; the deviation between them is small and vanishes with
mesh refinement, see [Assembling a model](../assembling.rst).

In [4]:
bk = np.zeros(N, dtype=bool)
ends = isclose(x, 0) | isclose(x, L)
for dof in (0, 3, 6):  # u, v, w
    bk[dof::DOF] = ends
bu = ~bk

Kuu = KC0[bu][:, bu]
Muu = M[bu][:, bu]

## The 16 lowest frequencies

In [5]:
eigvals, eigvecs = lowest_frequencies(Kuu, Muu, 16)
omegan = np.sqrt(eigvals)
for i, om in enumerate(omegan):
    print(f'mode {i+1:2d}: omega = {om:9.3f} rad/s, f = {om/(2*pi):8.3f} Hz')

mode  1: omega =  1096.073 rad/s, f =  174.445 Hz
mode  2: omega =  1096.073 rad/s, f =  174.445 Hz
mode  3: omega =  1104.677 rad/s, f =  175.815 Hz
mode  4: omega =  1104.677 rad/s, f =  175.815 Hz
mode  5: omega =  1172.827 rad/s, f =  186.661 Hz
mode  6: omega =  1172.827 rad/s, f =  186.661 Hz
mode  7: omega =  1219.385 rad/s, f =  194.071 Hz
mode  8: omega =  1219.385 rad/s, f =  194.071 Hz
mode  9: omega =  1313.437 rad/s, f =  209.040 Hz
mode 10: omega =  1313.437 rad/s, f =  209.040 Hz
mode 11: omega =  1459.079 rad/s, f =  232.220 Hz
mode 12: omega =  1459.079 rad/s, f =  232.220 Hz
mode 13: omega =  1500.921 rad/s, f =  238.879 Hz
mode 14: omega =  1500.921 rad/s, f =  238.879 Hz
mode 15: omega =  1724.313 rad/s, f =  274.433 Hz
mode 16: omega =  1724.313 rad/s, f =  274.433 Hz


The frequencies come in pairs. The first four modes, $w$ on the developed
mid-surface, one of each pair:

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(9, 5), sharex=True, sharey=True)
for ax, i in zip(axes.flat, (0, 2, 4, 6)):
    mode = np.zeros(N)
    mode[bu] = eigvecs[:, i]
    w = mode[6::DOF].reshape(nx, ny)
    ax.contourf(ymesh, xmesh, w/np.abs(w).max(), levels=30, cmap='coolwarm')
    ax.set_title(f'mode {i+1}, {omegan[i]/(2*pi):.1f} Hz')
    ax.set_aspect('equal')
for ax in axes[-1]:
    ax.set_xlabel('$y$ (m)')
for ax in axes[:, 0]:
    ax.set_ylabel('$x$ (m)')
fig.tight_layout()
plt.show()

## Comparison

The reference is the regression value stored in
`tests/test_natural_frequency.py`, the lowest angular frequency of the same
model. The Donnell-type kinematics lose accuracy for modes with few
circumferential waves [Leissa1973]; for those modes the Sanders-type
element, `bfsccylinder.sanders`, is the safer choice, see
[Kinematics of the cylindrical shell](../kinematics.rst).

In [7]:
ref = 1096.10818933  # rad/s
table([{'result': 'omega_1 (rad/s)', 'value': omegan[0], 'reference': ref,
               'source': 'regression value of the test',
               'rel. diff. (%)': 100*(omegan[0] - ref)/ref}])

| result | value | reference | source | rel. diff. (%) |
|---|---|---|---|---|
| omega_1 (rad/s) | 1096.1 | 1096.1 | regression value of the test | -0.0031966 |

## References

- **[Leissa1973]** Leissa, A. W., Vibration of Shells, NASA SP-288, National Aeronautics and
  Space Administration, Washington, DC, 1973.